# Tie-Out — Every Reported Number

Collects the numbers the paper reports and checks that they trace to the code in this commit. Trains
nothing: it reads `results/*.json` and `output/pred_*` written by notebooks `00` to `09`. Everything here
is on the stratified group split; the original random split appears only in `11_split_comparison`.

**Run order**: `00` → `01` → … → `09` → this notebook → `11_split_comparison`.

**Checks, before any number is shown**
1. Every expected results file exists and none was produced from a dirty working tree.
2. For each result, the code it depends on is identical between the commit that produced it and the current
   commit: `sybil_pipeline.py`, `requirements.txt` and `data/` byte for byte, and the code cells of its own
   notebook (plus the search notebook, and for `06` the notebooks whose predictions it reads)
   (`sp.provenance`). Saved outputs and markdown may differ.
3. `03`, `04` and `06` used exactly the hyperparameters selected by `02_hyperparameter_search`.
4. The individual XGBoost and LightGBM metrics in `06` equal those of `03` and `04`.
5. Each model's F1 recomputed from its saved test predictions equals the saved metric.

In [1]:
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix
import sybil_pipeline as sp
pd.set_option('display.width', 220)

NOTEBOOKS = ['00_data_pipeline', '01_ablation_gini', '02_hyperparameter_search', '03_xgboost_sybil',
             '04_lightgbm_sybil', '05_logistic_regression_sybil', '06_cross_ensemble_sybil',
             '07_sensitivity_label_vintage', '08_ablation_families', '09_shap_importance']
R = {nb: sp.load_results(nb) for nb in NOTEBOOKS}
prov = sp.provenance(NOTEBOOKS)
print(prov.to_string(index=False))
assert prov['dependencies_unchanged'].all(), 'Some results were produced by different code: rerun them'
print('\nAll results trace to the code in this commit ✓')

                    notebook  commit  dependencies_unchanged
            00_data_pipeline 21adb3c                    True
            01_ablation_gini 21adb3c                    True
    02_hyperparameter_search 21adb3c                    True
            03_xgboost_sybil 21adb3c                    True
           04_lightgbm_sybil 21adb3c                    True
05_logistic_regression_sybil 21adb3c                    True
     06_cross_ensemble_sybil 21adb3c                    True
07_sensitivity_label_vintage 21adb3c                    True
        08_ablation_families 21adb3c                    True
          09_shap_importance 21adb3c                    True

All results trace to the code in this commit ✓


In [2]:
# ── Hyperparameters used = hyperparameters selected on validation ──
HP = R['02_hyperparameter_search']
assert R['03_xgboost_sybil']['params'] == HP['xgb_selected']
assert R['04_lightgbm_sybil']['params'] == HP['lgbm_selected']
assert R['06_cross_ensemble_sybil']['params'] == dict(xgb=HP['xgb_selected'], lgbm=HP['lgbm_selected'])
print('XGBoost  (selected on validation):', HP['xgb_selected'])
print('LightGBM (selected on validation):', HP['lgbm_selected'])
print('03, 04, 06 used these ✓')

# ── 06 individual models = 03 / 04 ──
ens = {r['name']: r for r in R['06_cross_ensemble_sybil']['models']}
for nb in ['03_xgboost_sybil', '04_lightgbm_sybil']:
    single = R[nb]['models'][0]
    for k in ['threshold', 'precision', 'recall', 'f1', 'auroc', 'ap']:
        assert abs(ens[single['name']][k] - single[k]) < 1e-9, (nb, k)
print('06 individual models match 03 and 04 ✓')

XGBoost  (selected on validation): {'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}
LightGBM (selected on validation): {'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0}
03, 04, 06 used these ✓
06 individual models match 03 and 04 ✓


## Dataset and partitions (paper Tables 1 and 5)

In [3]:
D = R['00_data_pipeline']
print(f"Addresses: {D['n_addresses']:,}  |  Sybil: {D['n_sybil']:,} ({D['n_sybil']/D['n_addresses']*100:.2f}%)")
print(f"Provision edges dropped by the snapshot cutoff: {D['provision_edges_after_cutoff']}")
fg = D['provision_trees']
print(f"Gas provision trees: {fg['n']:,}  |  multi-wallet: {fg['multi_wallet']:,}  |  largest: {fg['largest']:,} wallets"
      f"  |  Sybils in multi-wallet trees: {fg['sybils_in_multi']:,}\n")
cat = pd.DataFrame(D['categories']).T
cat['pct_of_total'] = (cat['n'] / cat['n'].sum() * 100).round(2)
cat['sybil_rate_pct'] = (cat['sybil'] / cat['n'] * 100).round(2)
print('Taxonomy categories:'); print(cat.to_string(), '\n')
print('Partitions (before upsampling):'); print(pd.DataFrame(D['splits']).T.to_string())
L = D['leakage']
print(f"\nRelational leakage (row overlap is asserted 0): test rows sharing a gas provision tree with train "
      f"{L['test_rows_sharing_tree_with_train']:,} of {L['test_rows']:,}; test Sybils sharing one with a train Sybil "
      f"{L['test_sybils_sharing_tree_with_train_sybil']:,} of {L['test_sybils']:,}")

Addresses: 434,786  |  Sybil: 18,211 (4.19%)
Provision edges dropped by the snapshot cutoff: 296
Gas provision trees: 371,764  |  multi-wallet: 22,952  |  largest: 1,627 wallets  |  Sybils in multi-wallet trees: 2,322

Taxonomy categories:
                  n  sybil  pct_of_total  sybil_rate_pct
IxE           93061    698         21.40            0.75
IxI           34391   1566          7.91            4.55
IxL          306658  15921         70.53            5.19
No provider     676     26          0.16            3.85 

Partitions (before upsampling):
               Total   Sybil  NonSybil  SybilPct
Train       213046.0  8924.0  204122.0      4.19
Validation   91305.0  3824.0   87481.0      4.19
Test        130435.0  5463.0  124972.0      4.19

Relational leakage (row overlap is asserted 0): test rows sharing a gas provision tree with train 0 of 130,435; test Sybils sharing one with a train Sybil 0 of 5,463


## Test-set metrics (paper Table 3)

Thresholds are the F1-maximizing thresholds on validation; the blend weight in `06` is also chosen on
validation. AP is average precision (area under the precision-recall curve). FPR is the false-positive
rate among true non-Sybils at the model's threshold.

In [4]:
MODELS = {'XGBoost (tuned, 3-seed)': ('03_xgboost_sybil', 'prob_xgb'),
          'LightGBM (tuned, 3-seed)': ('04_lightgbm_sybil', 'prob_lgbm'),
          'Logistic Regression (C=0.01, L1)': ('05_logistic_regression_sybil', 'prob_lr'),
          'Cross-Ensemble': ('06_cross_ensemble_sybil', 'prob_ens')}

def result(name):
    nb, _ = MODELS[name]
    return next(x for x in R[nb]['models'] if x['name'] == name)

def preds(name):
    nb, col = MODELS[name]
    p = sp.load_predictions(nb, 'test')
    return p[['addr', 'category', 'y']].assign(prob=p[col])

rows = []
for name in MODELS:
    r, p = result(name), preds(name)
    yhat = (p['prob'] >= r['threshold']).astype(int)
    assert abs(f1_score(p['y'], yhat) - r['f1']) < 1e-9, name   # predictions match saved metrics
    tn, fp, fn, tp = confusion_matrix(p['y'], yhat).ravel()
    rows.append(dict(Model=name, **{k: r[k] for k in ['precision', 'recall', 'f1', 'auroc', 'ap', 'threshold']},
                     fpr=fp / (fp + tn), fp=int(fp)))
table = pd.DataFrame(rows).set_index('Model')
print(table.round(4).to_string())
E = R['06_cross_ensemble_sybil']
print(f"\nBlend: XGB weight = {E['xgb_weight']:.2f} (LightGBM {1 - E['xgb_weight']:.2f})  |  "
      f"disagreement at 0.5: {E['disagreement_n']:,} ({E['disagreement_pct']:.2f}%)  |  "
      f"rounds XGB {E['rounds']['xgb']}  LGBM {E['rounds']['lgbm']}")

                                  precision  recall      f1   auroc      ap  threshold     fpr     fp
Model                                                                                                
XGBoost (tuned, 3-seed)              0.7196  0.7205  0.7200  0.9657  0.7645     0.5824  0.0123   1534
LightGBM (tuned, 3-seed)             0.7267  0.7111  0.7188  0.9683  0.7696     0.6001  0.0117   1461
Logistic Regression (C=0.01, L1)     0.1499  0.6167  0.2412  0.8368  0.1656     0.6823  0.1529  19104
Cross-Ensemble                       0.6971  0.7388  0.7173  0.9683  0.7699     0.5035  0.0140   1754

Blend: XGB weight = 0.06 (LightGBM 0.94)  |  disagreement at 0.5: 703 (0.54%)  |  rounds XGB [1679, 1617, 1578]  LGBM [1277, 1248, 1143]


## Per-category metrics (paper Table 6)

Each model at its own validation-selected threshold, evaluated within each taxonomy category of the test set.

In [5]:
per_cat = []
for name in MODELS:
    r, p = result(name), preds(name)
    per_cat.append(sp.metrics_by_category(p['y'], p['prob'], p['category'], r['threshold']).assign(Model=name))
print(pd.concat(per_cat).set_index(['Model', 'category']).round(4).to_string())

                                                  n  sybil  precision  recall      f1   auroc     fpr
Model                            category                                                            
XGBoost (tuned, 3-seed)          IxE          28171    258     0.7885  0.1589  0.2645  0.9053  0.0004
                                 IxI           9967    426     0.9691  0.3685  0.5340  0.9209  0.0005
                                 IxL          92079   4770     0.7108  0.7822  0.7448  0.9746  0.0174
                                 No provider    218      9     1.0000  0.7778  0.8750  0.9628  0.0000
LightGBM (tuned, 3-seed)         IxE          28171    258     0.8333  0.1744  0.2885  0.9261  0.0003
                                 IxI           9967    426     0.6207  0.3803  0.4716  0.9155  0.0104
                                 IxL          92079   4770     0.7307  0.7696  0.7496  0.9762  0.0155
                                 No provider    218      9     1.0000  0.7778  0.8

## Hyperparameter search (paper Tables 7 and 8; validation only)

In [6]:
for model in ['xgboost', 'lightgbm']:
    t = pd.read_csv(f'results/02_search_{model}.csv')
    print(f'{model}: {len(t)} configurations; top 5 by validation F1')
    print(t.sort_values(['val_f1', 'val_ap'], ascending=False).head(5).round(4).to_string(index=False), '\n')

xgboost: 74 configurations; top 5 by validation F1
 learning_rate  max_depth  subsample  colsample_bytree  min_child_weight  val_f1  val_ap  val_logloss  val_threshold  rounds  seconds
          0.05          8        0.7               1.0                 1  0.7032  0.7552       0.0826         0.5262    1679     90.3
          0.05         10        0.5               1.0                 1  0.7025  0.7527       0.0820         0.5450     906     69.6
          0.05         12        0.5               0.8                 1  0.7019  0.7543       0.0805         0.5808     547     55.6
          0.05          8        0.7               1.0                10  0.7016  0.7518       0.0840         0.5573    2536    134.2
          0.10         12        0.7               0.8                 1  0.7012  0.7525       0.0824         0.5899     297     29.0 

lightgbm: 48 configurations; top 5 by validation F1
 num_leaves  learning_rate  subsample  reg_lambda  val_f1  val_ap  val_logloss  val_thresho

## Robustness checks (group split, 10 split seeds unless noted)

- `01`: keep-or-drop test for the corrected `gini_coefficient` (rule fixed in advance; validation only).
- `07`: labels as known before the snapshot vs current labels.
- `08`: each feature family removed, and each family alone (report only).
- `09`: mean |SHAP| by feature family and taxonomy category (test partition, split seed 42).

In [7]:
G = R['01_ablation_gini']
print(f"Gini (01): removing it lowered validation F1 on {G['splits_lowered']} of {len(G['per_split'])} splits "
      f"(one-sided sign test p = {G['sign_test_p']:.3f}; mean change {G['mean_f1_change']:+.4f}). Decision: {G['decision']}")
V = R['07_sensitivity_label_vintage']
print(f"\nLabel vintage (07): labeled addresses in the network, current {V['labeled_current']:,}, "
      f"pre-snapshot {V['labeled_presnapshot']:,}; wallets with any feature changed {V['wallets_changed']:,}")
print('Pre-snapshot minus current, mean ± SD over splits: ' +
      ', '.join(f"{k} {V['diff_mean'][k]:+.4f} ± {V['diff_sd'][k]:.4f}" for k in V['diff_mean']))
F = pd.DataFrame(R['08_ablation_families']['summary']).T
F = F[['n_features_mean', 'val_f1_mean', 'test_f1_mean', 'test_f1_std', 'test_ap_mean', 'test_auroc_mean',
       'test_f1_change_vs_all_mean']]
print('\nFeature families (08), mean and SD over splits:'); print(F.round(4).to_string())
Sh = pd.DataFrame(R['09_shap_importance']['mean_abs_shap'])
print('\nSum of mean |SHAP| by family (09):')
print(Sh.groupby('family')[['All', 'IxL', 'IxI', 'IxE']].sum().sort_values('All', ascending=False).round(4).to_string())

Gini (01): removing it lowered validation F1 on 3 of 10 splits (one-sided sign test p = 0.945; mean change +0.0011). Decision: remove

Label vintage (07): labeled addresses in the network, current 4,308, pre-snapshot 4,115; wallets with any feature changed 10,575
Pre-snapshot minus current, mean ± SD over splits: val_f1 +0.0089 ± 0.0115, test_f1 -0.0047 ± 0.0070, test_ap -0.0055 ± 0.0090, test_auroc -0.0023 ± 0.0039

Feature families (08), mean and SD over splits:
                                        n_features_mean  val_f1_mean  test_f1_mean  test_f1_std  test_ap_mean  test_auroc_mean  test_f1_change_vs_all_mean
All features                                       62.0       0.7121        0.7131       0.0060        0.7607           0.9682                      0.0000
Without LayerZero transactions                     40.0       0.6489        0.6472       0.0046        0.6718           0.9366                     -0.0659
Without Ethereum transactions                      53.0       0.69

In [8]:
# ── Markdown table for the README / paper ─────────────────────
lines = ['| Model | Precision | Recall | F1 | AUROC | AP | FPR |', '|---|---|---|---|---|---|---|']
for model, row in table.iterrows():
    lines.append(f"| {model} | {row['precision']:.3f} | {row['recall']:.3f} | {row['f1']:.3f} | "
                 f"{row['auroc']:.3f} | {row['ap']:.3f} | {row['fpr']:.4f} |")
print('\n'.join(lines))
print(f"\nSource: results/ checked against commit {sp.CODE_COMMIT}")

| Model | Precision | Recall | F1 | AUROC | AP | FPR |
|---|---|---|---|---|---|---|
| XGBoost (tuned, 3-seed) | 0.720 | 0.720 | 0.720 | 0.966 | 0.764 | 0.0123 |
| LightGBM (tuned, 3-seed) | 0.727 | 0.711 | 0.719 | 0.968 | 0.770 | 0.0117 |
| Logistic Regression (C=0.01, L1) | 0.150 | 0.617 | 0.241 | 0.837 | 0.166 | 0.1529 |
| Cross-Ensemble | 0.697 | 0.739 | 0.717 | 0.968 | 0.770 | 0.0140 |

Source: results/ checked against commit 21adb3c
